##Customer Churn & Retention Analysis (SQL)

### Project Overview:
VS Studio is a subscription-based digital content platform. This project uses SQL to analyze customer subscriptions, payments, revenue, churn, and retention metrics to evaluate business performance and identify opportunities for growth. The analysis demonstrates how data can be transformed into actionable business insights that support revenue optimization and customer retention strategies.

### Business Objectives:
- Analyze monthly revenue and identify revenue trends.
- Compare financial performance across subscription plans.
- Calculate customer and subscription churn rates.
- Measure Monthly Recurring Revenue (MRR).
- Identify high-value customers and revenue contributors.
- Evaluate customer retention over time.
- Generate data-driven recommendations to improve revenue and reduce churn.

### Tools & Technologies:
- SQL (SQLite)
- Python
- Pandas
- Google Colab
- Data Visualization

### Dataset:
The project uses a subscription-business database consisting of three relational tables:

#### Customers:
- Customer ID
- Signup Date
- Country

#### Subscriptions:
- Subscription ID
- Customer ID
- Subscription Plan
- Monthly Price
- Start Date
- End Date
- Subscription Status

#### Payments:
- Payment ID
- Subscription ID
- Payment Date
- Payment Amount
- Payment Status

### SQL Skills Demonstrated:
- SELECT Statements
- Filtering with WHERE
- GROUP BY and ORDER BY
- INNER JOIN Operations
- Aggregate Functions
- CASE WHEN Logic
- Common Table Expressions (CTEs)
- Window Functions (RANK, DENSE_RANK, LAG)
- Revenue Analysis
- Churn Analysis
- Retention Analysis

### Key Business Questions Answered:
- Which subscription plan generates the most revenue?
- What is the current Monthly Recurring Revenue (MRR)?
- Which customers contribute the highest lifetime value?
- What percentage of subscribers have churned?
- How effectively does the business retain customers month over month?
- What trends exist in successful versus failed payments?

### Business Impact:
This project demonstrates the ability to use SQL and data analytics techniques to evaluate customer behavior, identify revenue opportunities, monitor churn, and support strategic decision-making. The findings can help a subscription-based business improve customer retention, optimize subscription offerings, and increase long-term profitability.

In [ ]:

import sqlite3
import pandas as pd

conn = sqlite3.connect(":memory:")

conn.executescript("""
CREATE TABLE customers (
    customer_id INTEGER PRIMARY KEY,
    signup_date DATE,
    country TEXT
);

CREATE TABLE subscriptions (
    subscription_id INTEGER PRIMARY KEY,
    customer_id INTEGER,
    plan TEXT,
    monthly_price REAL,
    start_date DATE,
    end_date DATE,
    status TEXT,
    FOREIGN KEY (customer_id) REFERENCES customers(customer_id)
);

CREATE TABLE payments (
    payment_id INTEGER PRIMARY KEY,
    subscription_id INTEGER,
    payment_date DATE,
    amount REAL,
    payment_status TEXT,
    FOREIGN KEY (subscription_id)
        REFERENCES subscriptions(subscription_id)
);
""")

print("Tables created successfully.")

Tables created successfully.


In [ ]:

conn.executescript("""
INSERT INTO customers VALUES
(1, '2026-01-05', 'USA'),
(2, '2026-01-12', 'Canada'),
(3, '2026-01-20', 'USA'),
(4, '2026-02-02', 'UK'),
(5, '2026-02-10', 'USA'),
(6, '2026-02-18', 'Canada'),
(7, '2026-03-01', 'UK'),
(8, '2026-03-08', 'USA');

INSERT INTO subscriptions VALUES
(101, 1, 'Basic',       10, '2026-01-05', NULL,         'active'),
(102, 2, 'Premium',    30, '2026-01-12', NULL,         'active'),
(103, 3, 'Basic',       10, '2026-01-20', '2026-02-20', 'cancelled'),
(104, 4, 'Standard',   20, '2026-02-02', NULL,         'active'),
(105, 5, 'Premium',    30, '2026-02-10', '2026-03-10', 'cancelled'),
(106, 6, 'Basic',       10, '2026-02-18', NULL,         'active'),
(107, 7, 'Standard',   20, '2026-03-01', NULL,         'active'),
(108, 8, 'Premium',    30, '2026-03-08', NULL,         'active');

INSERT INTO payments VALUES
(1001, 101, '2026-01-05', 10, 'paid'),
(1002, 102, '2026-01-12', 30, 'paid'),
(1003, 103, '2026-01-20', 10, 'paid'),
(1004, 101, '2026-02-05', 10, 'paid'),
(1005, 102, '2026-02-12', 30, 'paid'),
(1006, 103, '2026-02-20', 10, 'paid'),
(1007, 104, '2026-02-02', 20, 'paid'),
(1008, 105, '2026-02-10', 30, 'paid'),
(1009, 106, '2026-02-18', 10, 'paid'),
(1010, 101, '2026-03-05', 10, 'paid'),
(1011, 102, '2026-03-12', 30, 'paid'),
(1012, 104, '2026-03-02', 20, 'paid'),
(1013, 105, '2026-03-10', 30, 'paid'),
(1014, 106, '2026-03-18', 10, 'paid'),
(1015, 107, '2026-03-01', 20, 'paid'),
(1016, 108, '2026-03-08', 30, 'paid');
""")

print("Sample data inserted successfully.")

Sample data inserted successfully.


## 1. Database Setup

In this section, I establish an SQLite database and create three relational tables:

- `customers` — stores customer information.
- `subscriptions` — stores subscription plans, pricing, and status.
- `payments` — stores payment transactions and payment status.

These tables will serve as the foundation for the customer churn, revenue, and retention analysis.

In [ ]:
def run_query(sql):
    return pd.read_sql_query(sql, conn)

## 2. Revenue Analysis

In this section, I analyze the company's revenue performance using payment data.

The analysis focuses on:
- Total revenue and average payment amount.
- Monthly revenue trends.
- Revenue generated by each subscription plan.

The goal is to identify revenue patterns and understand which subscription plans contribute most to the business.

#### 2.1 Overall Revenue Performance

This query calculates the total number of successful payments, total revenue generated, and average payment amount.

It provides a high-level overview of the company's payment performance.

In [ ]:
run_query("""
SELECT
    COUNT(*) AS payment_count,
    SUM(amount) AS total_revenue,
    AVG(amount) AS average_payment
FROM payments
WHERE payment_status = 'paid';
""")

,payment_count,total_revenue,average_payment
0,16,310.0,19.375


#### 2.2 Monthly Revenue Trends

This query calculates total revenue generated during each month using successful payments.

It groups payment records by month and organizes the results chronologically.

The goal is to identify revenue trends and understand how the company's revenue changes over time.

In [ ]:
run_query("""
SELECT
    strftime('%Y-%m', payment_date) AS payment_month,
    SUM(amount) AS monthly_revenue
FROM payments
WHERE payment_status = 'paid'
GROUP BY strftime('%Y-%m', payment_date)
ORDER BY payment_month ASC;
""")

,payment_month,monthly_revenue
0,2026-01,50.0
1,2026-02,110.0
2,2026-03,150.0


#### 2.3 Revenue by Subscription Plan

This query combines the payments and subscriptions tables using a JOIN to calculate total revenue generated by each subscription plan.

The results are grouped by subscription plan and sorted from highest to lowest revenue.

The goal is to identify which subscription plans contribute the most to the company's total revenue.

In [ ]:
run_query("""
SELECT
    s.plan,
    SUM(p.amount) AS total_revenue
FROM payments p
JOIN subscriptions s
    ON p.subscription_id = s.subscription_id
WHERE p.payment_status = 'paid'
GROUP BY s.plan
ORDER BY total_revenue DESC;
""")

,plan,total_revenue
0,Premium,180.0
1,Basic,70.0
2,Standard,60.0


## 3. Subscription & Churn Analysis

In this section, I examine subscription activity and customer cancellations to understand customer churn.

The analysis focuses on:
- Counting active subscriptions by plan.
- Calculating the overall subscription churn rate.
- Identifying monthly recurring revenue (MRR).
- Comparing churn rates across subscription plans.
- Tracking customer cancellation patterns.

The goal is to identify subscription trends and understand which plans may need improvements in customer retention.

#### 3.1 Active Subscriptions by Plan

This query counts the number of active subscriptions for each subscription plan.

The results are grouped by plan and sorted from the highest to lowest number of active subscriptions.

The goal is to understand which subscription plans have the most active customers.

In [ ]:
run_query("""
SELECT
    plan,
    COUNT(*) AS subscription_count
FROM subscriptions
WHERE status = 'active'
GROUP BY plan
ORDER BY subscription_count DESC;
""")

,plan,subscription_count
0,Standard,2
1,Premium,2
2,Basic,2


#### 3.2 Overall Subscription Churn Rate

This query calculates the overall subscription churn rate by counting the total number of subscriptions and identifying how many have been cancelled.

A CASE WHEN statement is used to count cancelled subscriptions.

The goal is to measure the proportion of subscriptions that have been cancelled and evaluate overall subscription performance.

In [ ]:
run_query("""
SELECT
    COUNT(*) AS total_subscriptions,
    SUM(
        CASE WHEN status = 'cancelled' THEN 1 ELSE 0 END
    ) AS cancelled_subscriptions,
    100.0 * SUM(
        CASE WHEN status = 'cancelled' THEN 1 ELSE 0 END
    ) / COUNT(*) AS churn_rate_pct
FROM subscriptions;
""")

,total_subscriptions,cancelled_subscriptions,churn_rate_pct
0,8,2,25.0


#### 3.3 Monthly Recurring Revenue (MRR)

This query calculates Monthly Recurring Revenue (MRR) at the end of January, February, and March.

It identifies subscriptions that were active at each month-end date and sums their monthly subscription prices.

The goal is to track recurring revenue over time and understand how active subscriptions contribute to the business's monthly revenue.

In [ ]:
run_query("""
WITH month_ends AS (
    SELECT '2026-01-31' AS month_end
    UNION ALL
    SELECT '2026-02-28'
    UNION ALL
    SELECT '2026-03-31'
)
SELECT
    m.month_end,
    SUM(s.monthly_price) AS mrr
FROM month_ends m
JOIN subscriptions s
    ON s.start_date <= m.month_end
    AND (s.end_date IS NULL OR s.end_date > m.month_end)
GROUP BY m.month_end
ORDER BY m.month_end;
""")

,month_end,mrr
0,2026-01-31,50.0
1,2026-02-28,100.0
2,2026-03-31,120.0


#### 3.4 Revenue per Active Customer

This query calculates monthly revenue and the number of active customers for each month.

It uses Common Table Expressions (CTEs) to organize the calculations and combines the results to determine revenue per active customer.

The goal is to understand how much revenue each active customer generates on average and how this metric changes over time.

In [ ]:
run_query("""
WITH monthly_revenue AS (
    SELECT
        strftime('%Y-%m', payment_date) AS payment_month,
        SUM(amount) AS revenue
    FROM payments
    WHERE payment_status = 'paid'
    GROUP BY strftime('%Y-%m', payment_date)
),

active_customers AS (
    SELECT
        '2026-01' AS payment_month,
        COUNT(DISTINCT c.customer_id) AS active_count
    FROM customers c
    JOIN subscriptions s
        ON c.customer_id = s.customer_id
    WHERE s.start_date <= '2026-01-31'
      AND (s.end_date IS NULL OR s.end_date > '2026-01-31')

    UNION ALL

    SELECT
        '2026-02',
        COUNT(DISTINCT c.customer_id)
    FROM customers c
    JOIN subscriptions s
        ON c.customer_id = s.customer_id
    WHERE s.start_date <= '2026-02-28'
      AND (s.end_date IS NULL OR s.end_date > '2026-02-28')

    UNION ALL

    SELECT
        '2026-03',
        COUNT(DISTINCT c.customer_id)
    FROM customers c
    JOIN subscriptions s
        ON c.customer_id = s.customer_id
    WHERE s.start_date <= '2026-03-31'
      AND (s.end_date IS NULL OR s.end_date > '2026-03-31')
)

SELECT
    r.payment_month,
    r.revenue,
    a.active_count,
    ROUND(r.revenue * 1.0 / a.active_count, 2) AS revenue_per_active_customer
FROM monthly_revenue r
JOIN active_customers a
    ON r.payment_month = a.payment_month
ORDER BY r.payment_month;
""")

,payment_month,revenue,active_count,revenue_per_active_customer
0,2026-01,50.0,3,16.67
1,2026-02,110.0,5,22.00
2,2026-03,150.0,6,25.00


#### 3.5 Monthly Subscription Cancellations

This query calculates the number of cancelled subscriptions for each month.

It groups cancelled subscriptions by their end date and sorts the results chronologically.

The goal is to identify when cancellations occur and understand monthly cancellation patterns.

In [ ]:
run_query("""
SELECT
    strftime('%Y-%m', end_date) AS cancellation_month,
    COUNT(subscription_id) AS cancelled_subscriptions
FROM subscriptions
WHERE status = 'cancelled'
GROUP BY strftime('%Y-%m', end_date)
ORDER BY cancellation_month;
""")

,cancellation_month,cancelled_subscriptions
0,2026-02,1
1,2026-03,1


#### 3.6 Churn Rate by Subscription Plan

This query calculates the churn rate for each subscription plan.

A Common Table Expression (CTE) is used to calculate the total number of subscriptions and cancelled subscriptions for each plan.

The results show the percentage of subscriptions that have been cancelled within each plan.

The goal is to identify which subscription plans experience the highest churn and may require improvements to customer retention.

In [ ]:
run_query("""
WITH plan_churn AS (
    SELECT
        plan,
        COUNT(*) AS total_subscriptions,
        SUM(
            CASE
                WHEN status = 'cancelled' THEN 1
                ELSE 0
            END
        ) AS cancelled_subscriptions
    FROM subscriptions
    GROUP BY plan
)

SELECT
    plan,
    total_subscriptions,
    cancelled_subscriptions,
    ROUND(
        100.0 * cancelled_subscriptions / total_subscriptions,
        2
    ) AS churn_rate_pct
FROM plan_churn
ORDER BY churn_rate_pct DESC;
""")

,plan,total_subscriptions,cancelled_subscriptions,churn_rate_pct
0,Basic,3,1,33.33
1,Premium,3,1,33.33
2,Standard,2,0,0.00


## 4. Customer Revenue & Value Analysis

In this section, I analyze customer spending behavior to identify the customers who contribute the most revenue.

The analysis focuses on:
- Calculating total revenue per customer.
- Ranking customers by total revenue.
- Identifying high-value customers.
- Understanding how revenue is distributed across customers.

The goal is to identify valuable customer segments and understand their contribution to overall business revenue.

#### 4.2 Ranking Customers by Revenue

This query uses a Common Table Expression (CTE) to calculate the total revenue generated by each customer.

The RANK() window function assigns a ranking to each customer based on their total revenue, from highest to lowest.

The goal is to identify the highest-revenue customers and demonstrate the use of SQL window functions for customer analysis.

In [ ]:
run_query("""
WITH customer_revenue AS (
    SELECT
        c.customer_id,
        SUM(p.amount) AS total_revenue
    FROM customers c
    JOIN subscriptions s
        ON c.customer_id = s.customer_id
    JOIN payments p
        ON s.subscription_id = p.subscription_id
    WHERE p.payment_status = 'paid'
    GROUP BY c.customer_id
)

SELECT
    customer_id,
    total_revenue,
    RANK() OVER (ORDER BY total_revenue DESC) AS revenue_rank
FROM customer_revenue
ORDER BY revenue_rank;
""")

,customer_id,total_revenue,revenue_rank
0,2,90.0,1
1,5,60.0,2
2,4,40.0,3
3,1,30.0,4
4,8,30.0,4
5,3,20.0,6
6,6,20.0,6
7,7,20.0,6


#### 4.3 Month-over-Month Revenue Growth

This query uses Common Table Expressions (CTEs) and the LAG() window function to compare monthly revenue with the previous month's revenue.

It calculates:
- Current monthly revenue.
- Previous month's revenue.
- Percentage change in revenue.

The goal is to identify revenue growth patterns and understand how the company's financial performance changes over time.

In [ ]:
run_query("""
WITH monthly_revenue AS (
    SELECT
        strftime('%Y-%m', payment_date) AS payment_month,
        SUM(amount) AS revenue
    FROM payments
    WHERE payment_status = 'paid'
    GROUP BY strftime('%Y-%m', payment_date)
),

revenue_with_previous AS (
    SELECT
        payment_month,
        revenue,
        LAG(revenue) OVER (ORDER BY payment_month) AS previous_revenue
    FROM monthly_revenue
)

SELECT
    payment_month,
    revenue,
    previous_revenue,
    ROUND(
        (revenue - previous_revenue) * 100.0 / previous_revenue,
        2
    ) AS pct_change
FROM revenue_with_previous
ORDER BY payment_month;
""")

,payment_month,revenue,previous_revenue,pct_change
0,2026-01,50.0,NaN,NaN
1,2026-02,110.0,50.0,120.00
2,2026-03,150.0,110.0,36.36


#### 4.1 Total Revenue per Customer

This query combines the customers, subscriptions, and payments tables using JOIN operations to calculate the total revenue generated by each customer.

It groups payment records by customer and sorts the results from highest to lowest revenue.

The goal is to identify which customers contribute the most revenue to the business.

In [ ]:
run_query("""
SELECT
    c.customer_id,
    SUM(p.amount) AS total_revenue
FROM customers c
JOIN subscriptions s
    ON c.customer_id = s.customer_id
JOIN payments p
    ON s.subscription_id = p.subscription_id
WHERE p.payment_status = 'paid'
GROUP BY c.customer_id
ORDER BY total_revenue DESC;
""")

,customer_id,total_revenue
0,2,90.0
1,5,60.0
2,4,40.0
3,8,30.0
4,1,30.0
5,7,20.0
6,6,20.0
7,3,20.0


## 5. Customer Retention Analysis

In this section, I analyze customer retention by tracking customers who make payments across consecutive months.

The analysis focuses on:
- Identifying monthly active customers.
- Measuring month-to-month customer retention.
- Identifying customers who return in consecutive months.
- Calculating monthly retention rates.
- Comparing customer activity across different months.

The goal is to understand customer loyalty and identify opportunities to improve long-term retention.

#### 5.1 January-to-February Customer Retention

This query identifies customers who made successful payments in both January and February.

A Common Table Expression (CTE) identifies monthly active customers, while a self-JOIN matches customers across the two months.

The query calculates:
- Number of retained customers.
- January-to-February retention rate.

The goal is to measure how many January customers continued making payments in February.

In [ ]:
run_query("""
WITH monthly_active AS (
    SELECT DISTINCT
        strftime('%Y-%m', p.payment_date) AS month,
        s.customer_id
    FROM payments p
    JOIN subscriptions s
        ON p.subscription_id = s.subscription_id
    WHERE p.payment_status = 'paid'
)

SELECT
    COUNT(DISTINCT previous.customer_id) AS retained_customers,
    COUNT(DISTINCT previous.customer_id) * 100.0 / 3 AS retention_rate_pct
FROM monthly_active previous
JOIN monthly_active current
    ON previous.customer_id = current.customer_id
    AND previous.month = '2026-01'
    AND current.month = '2026-02';
""")

,retained_customers,retention_rate_pct
0,3,100.0


#### 5.2 Identifying Retained Customers

This query identifies the specific customers who made successful payments in both January and February.

It uses a Common Table Expression (CTE) to identify monthly active customers and a self-JOIN to match customers across the two months.

The results display:
- The previous month.
- The current month.
- The customer ID of each retained customer.

The goal is to identify which customers continued their subscriptions from one month to the next and examine customer-level retention.

In [ ]:
run_query("""
WITH monthly_active AS (
    SELECT DISTINCT
        strftime('%Y-%m', p.payment_date) AS month,
        s.customer_id
    FROM payments p
    JOIN subscriptions s
        ON p.subscription_id = s.subscription_id
    WHERE p.payment_status = 'paid'
)

SELECT
    previous.month AS previous_month,
    current.month AS current_month,
    previous.customer_id
FROM monthly_active previous
JOIN monthly_active current
    ON previous.customer_id = current.customer_id
    AND current.month = '2026-02'
    AND previous.month = '2026-01'
ORDER BY previous.customer_id;
""")

,previous_month,current_month,customer_id
0,2026-01,2026-02,1
1,2026-01,2026-02,2
2,2026-01,2026-02,3


#### 5.3 Month-to-Month Retained Customers

This query calculates the number of customers who made successful payments in consecutive months.

It uses a Common Table Expression (CTE) to identify monthly active customers and a self-JOIN to match customers between consecutive months.

Unlike the previous queries, this query uses date calculations to automatically identify the following month instead of comparing only January and February.

The goal is to track customer retention across multiple consecutive months.

In [ ]:
run_query("""
WITH monthly_active AS (
    SELECT DISTINCT
        strftime('%Y-%m', p.payment_date) AS month,
        s.customer_id
    FROM payments p
    JOIN subscriptions s
        ON p.subscription_id = s.subscription_id
    WHERE p.payment_status = 'paid'
)

SELECT
    previous.month AS previous_month,
    current.month AS current_month,
    COUNT(DISTINCT previous.customer_id) AS retained_customers
FROM monthly_active previous
JOIN monthly_active current
    ON previous.customer_id = current.customer_id
    AND current.month = strftime(
        '%Y-%m',
        date(previous.month || '-01', '+1 month')
    )
GROUP BY previous.month, current.month
ORDER BY previous.month;
""")

,previous_month,current_month,retained_customers
0,2026-01,2026-02,3
1,2026-02,2026-03,5


#### 5.4 Month-to-Month Customer Retention Rate

This query calculates the customer retention rate between consecutive months.

It uses two Common Table Expressions (CTEs):
- `monthly_active` identifies customers who made successful payments each month.
- `retention` calculates the number of customers who returned in the following month.

The final query calculates:
- Previous month's active customers.
- Number of retained customers.
- Monthly customer retention rate.

The goal is to measure how effectively the business retains customers over time and identify changes in customer loyalty.

In [ ]:
run_query("""
WITH monthly_active AS (
    SELECT DISTINCT
        strftime('%Y-%m', p.payment_date) AS month,
        s.customer_id
    FROM payments p
    JOIN subscriptions s
        ON p.subscription_id = s.subscription_id
    WHERE p.payment_status = 'paid'
),

retention AS (
    SELECT
        previous.month AS previous_month,
        current.month AS current_month,
        COUNT(DISTINCT previous.customer_id) AS retained_customers
    FROM monthly_active previous
    JOIN monthly_active current
        ON previous.customer_id = current.customer_id
        AND current.month = strftime(
            '%Y-%m',
            date(previous.month || '-01', '+1 month')
        )
    GROUP BY previous.month, current.month
)

SELECT
    previous_month,
    current_month,
    (
        SELECT COUNT(DISTINCT customer_id)
        FROM monthly_active
        WHERE month = retention.previous_month
    ) AS previous_customers,
    retained_customers,
    ROUND(
        retained_customers * 100.0 /
        (
            SELECT COUNT(DISTINCT customer_id)
            FROM monthly_active
            WHERE month = retention.previous_month
        ),
        2
    ) AS retention_rate_pct
FROM retention
ORDER BY previous_month;
""")


,previous_month,current_month,previous_customers,retained_customers,retention_rate_pct
0,2026-01,2026-02,3,3,100.00
1,2026-02,2026-03,6,5,83.33


## 6. Business Insights & Recommendations

### 6.1 Customer Retention Insights

The customer retention analysis revealed the following findings:

- **January to February:** Customer retention was 100%, with all 3 active customers making successful payments in both months.
- **February to March:** Customer retention decreased to 83.33%, with 5 out of 6 active customers returning in March.
- **Retention trend:** Customer retention declined by 16.67 percentage points between the two periods.

These results suggest that while customer retention was initially strong, the business experienced a decrease in returning customers during the following month.

### 6.2 Business Recommendations

1. **Improve customer retention:** Investigate why customers stop making payments and consider targeted retention strategies.
2. **Monitor subscription churn:** Track cancellation rates across subscription plans to identify which plans experience higher churn.
3. **Focus on high-value customers:** Identify customers who generate the most revenue and consider loyalty incentives to encourage continued subscriptions.
4. **Monitor monthly revenue:** Review monthly revenue trends to identify potential business risks and opportunities.
5. **Strengthen customer engagement:** Consider promotional offers, personalized communication, and subscription renewal reminders to encourage customers to remain active.

### 6.3 Conclusion

This project demonstrates how SQL can be used to analyze subscription revenue, customer behavior, subscription churn, and customer retention.

By using SQL joins, aggregate functions, Common Table Expressions (CTEs), and window functions, the analysis identifies important business metrics and customer trends.

The findings can help a subscription-based business better understand its revenue performance, identify customer retention challenges, and make more informed business decisions.